# Retail Customer Sales | Exploratory Data Analysis

This notebook examines the supplied retail transactions from 2023 and 2024. It records the cleaning choices, checks each variable, compares sales across customer and product groups, and exports the analysis dataset. The Streamlit app (`Project1_EDA.py`) provides the same analysis with filters.

Install the packages in `requirements.txt` before running. Keep the notebook beside `retail_customer_sales_dataset.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path("retail_customer_sales_dataset.csv")
df = pd.read_csv(DATA)
print("Shape:", df.shape)
display(df.head(), df.tail(), df.dtypes.rename("dtype").to_frame())


## Data quality audit

Missing-value percentages use all source records as the denominator. Exact repeated rows are counted before removing later copies. Gender and Region labels are listed so the normalization rules can be reviewed.


In [ ]:
missing = pd.DataFrame({"missing_count": df.isna().sum(), "missing_percent": 100 * df.isna().mean()})
display(missing[missing.missing_count.gt(0)].sort_values("missing_percent", ascending=False))
print("Exact duplicate rows:", int(df.duplicated().sum()))
for col in ["Gender", "Region"]:
    display(df[col].value_counts(dropna=False).rename_axis(col).to_frame("count"))


## Cleaning decisions

The source table is kept in `df`. `duplicates` contains later full-row copies; `work` contains the first copy for each row. Missing values are left blank. Impossible bounded values are flagged and masked for analysis. IQR outliers remain in the data because the screen cannot tell a real large transaction from a data error.


In [ ]:
all_rows = df.copy()
all_rows["_Source_Row"] = np.arange(2, len(all_rows) + 2)
all_rows["Flag_Duplicate"] = all_rows.duplicated(subset=df.columns.tolist(), keep="first")
duplicates = all_rows.loc[all_rows.Flag_Duplicate].copy()
work = all_rows.loc[~all_rows.Flag_Duplicate].copy()

for col in ["Order_Date", "Delivery_Date"]:
    work[col + "_Raw"] = work[col].astype("string")
    work[col] = pd.to_datetime(work[col], errors="coerce", format="mixed")
work["Flag_Date_Format_Order"] = work["Order_Date_Raw"].notna() & ~work["Order_Date_Raw"].str.strip().str.fullmatch(r"\d{4}-\d{2}-\d{2}")
work["Flag_Date_Format_Delivery"] = work["Delivery_Date_Raw"].notna() & ~work["Delivery_Date_Raw"].str.strip().str.fullmatch(r"\d{4}-\d{2}-\d{2}")

gender_map = {"male":"Male", "m":"Male", "female":"Female", "f":"Female"}
work["Gender"] = work.Gender.astype("string").str.strip().str.casefold().map(gender_map).fillna(work.Gender.astype("string").str.strip().str.title())
region_map = {"north":"North", "nort":"North", "south":"South", "sth":"South", "east":"East", "west":"West"}
work["Region"] = work.Region.astype("string").str.strip().str.casefold().map(region_map).fillna(work.Region.astype("string").str.strip().str.title())

for col in ["Age", "Income", "Quantity", "Unit_Price", "Discount", "Sales", "Cost", "Profit", "Customer_Rating"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")
work["Flag_Invalid_Age"] = work.Age.notna() & ~work.Age.between(0, 120)
work.loc[work.Flag_Invalid_Age, "Age"] = np.nan
work["Flag_Invalid_Discount"] = work.Discount.notna() & ~work.Discount.between(0, .5)
work.loc[work.Flag_Invalid_Discount, "Discount"] = np.nan
work["Flag_Invalid_Rating"] = work.Customer_Rating.notna() & ~work.Customer_Rating.between(1, 5)
work.loc[work.Flag_Invalid_Rating, "Customer_Rating"] = np.nan
work["Flag_Invalid_Income"] = work.Income.notna() & (work.Income < 0)
work.loc[work.Flag_Invalid_Income, "Income"] = np.nan
work["Flag_Invalid_Quantity"] = work.Quantity.notna() & (work.Quantity <= 0)
work.loc[work.Flag_Invalid_Quantity, "Quantity"] = np.nan
work["Flag_Invalid_Unit_Price"] = work.Unit_Price.notna() & (work.Unit_Price <= 0)
work.loc[work.Flag_Invalid_Unit_Price, "Unit_Price"] = np.nan
work["Flag_Negative_Sales"] = work.Sales.notna() & (work.Sales < 0)
work.loc[work.Flag_Negative_Sales, "Sales"] = np.nan
work["Flag_Date_Order"] = work.Order_Date.isna()
work["Flag_Date_Delivery"] = work.Delivery_Date.isna()
work["Flag_Delivery_Before_Order"] = work.Delivery_Date.notna() & work.Order_Date.notna() & (work.Delivery_Date < work.Order_Date)
work.loc[work.Flag_Delivery_Before_Order, "Delivery_Date"] = pd.NaT

work["Profit_Margin"] = work.Profit.div(work.Sales.where(work.Sales.ne(0)))
work["Delivery_Days"] = (work.Delivery_Date - work.Order_Date).dt.days
work["Revenue_per_Customer"] = work.groupby("Customer_ID").Sales.transform("sum")
display(work.head())


## Descriptive statistics and distributions
The mode shown is the first mode when ties occur. Standard deviation and variance use pandas sample estimates. Skewness is a descriptive signal; it does not by itself indicate an error.

In [ ]:
numeric = ["Age", "Income", "Quantity", "Unit_Price", "Discount", "Sales", "Cost", "Profit"]
rows = []
for col in numeric:
    s = work[col].dropna(); mode = s.mode()
    rows.append({"Variable":col, "N":len(s), "Mean":s.mean(), "Median":s.median(), "Mode":mode.iloc[0] if len(mode) else np.nan, "Std":s.std(), "Variance":s.var(), "Q1":s.quantile(.25), "Q3":s.quantile(.75), "IQR":s.quantile(.75)-s.quantile(.25), "P5":s.quantile(.05), "P95":s.quantile(.95), "Skew":s.skew()})
stats = pd.DataFrame(rows).set_index("Variable")
display(stats)
for col in ["Age", "Income", "Sales", "Profit"]:
    fig, ax = plt.subplots(figsize=(7, 3))
    sns.histplot(work[col].dropna(), kde=True, ax=ax)
    ax.set_title(f"{col} distribution; skew={work[col].skew():.2f}")
    plt.show()


In [ ]:
for col in ["Sales", "Profit", "Age", "Income"]:
    s = work[col].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3-q1
    flag = s.lt(q1-1.5*iqr) | s.gt(q3+1.5*iqr)
    print(f"{col}: {int(flag.sum())} IQR review candidates of {len(s)} valid values")


## Categorical summaries and business questions
Grouped results describe this dataset; group size and transaction mix matter when interpreting averages.

In [ ]:
for col in ["Region", "Product_Category", "Customer_Segment"]:
    freq = work[col].value_counts(dropna=False).rename("count").to_frame()
    freq["proportion"] = freq["count"] / freq["count"].sum()
    display(freq)

questions = {
    "Average order value by region": work.groupby("Region").Sales.agg(["count", "mean", "sum"]).sort_values("mean", ascending=False),
    "Profit margin by category": work.groupby("Product_Category").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).assign(Profit_Margin=lambda x: x.Profit/x.Sales).sort_values("Profit_Margin"),
    "Sales by customer segment": work.groupby("Customer_Segment").Sales.agg(["count", "sum", "mean"]).sort_values("sum", ascending=False),
    "Average order value by payment method": work.groupby("Payment_Method").Sales.agg(["count", "mean"]).sort_values("mean", ascending=False),
    "Return rate by category": work.assign(returned=work.Returned.str.casefold().eq("yes")).groupby("Product_Category").returned.agg(["count", "mean"]).sort_values("mean", ascending=False),
    "Delivery duration by mode": work.groupby("Delivery_Mode").Delivery_Days.agg(["count", "mean", "median"]).sort_values("mean"),
}
for question, result in questions.items():
    print(question); display(result)


## Export the analysis files

The source CSV is unchanged. The cleaned export excludes exact duplicates and carries flags and engineered fields. Separate files keep the duplicate records and the source missing-value audit available for review.


In [ ]:
work.to_csv("retail_sales_cleaned.csv", index=False, encoding="utf-8-sig")
missing.to_csv("data_quality_audit.csv", index=False, encoding="utf-8-sig")
duplicates.to_csv("duplicate_rows.csv", index=False, encoding="utf-8-sig")
